# ***RUNNER WINNING PREDICTION***

**IMPORTING LIBRARIES AND UPLOADING THE DATASET**

In [ ]:
!pip install xgboost openpyxl shap fpdf --quiet

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

from google.colab import files
uploaded = files.upload()

df = pd.read_excel(next(iter(uploaded)))

df.head()

**DATA PREPROCESSING, TRAINING AND EVALUATION**

In [ ]:
df.dropna(inplace=True)

X = df.drop('Outcome', axis=1)
y = df['Outcome']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = XGBClassifier(eval_metric='logloss')
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f"\u2705 Model trained with accuracy: {accuracy * 100:.2f}%")

**PREDICTION AND REPORT GENERATION**

In [ ]:
# Runner Data Input & Prediction
print("\n\U0001f4cb Enter runner data below:")

user_input = []
for col in X.columns:
    val = float(input(f"{col}: "))
    user_input.append(val)

user_df = pd.DataFrame([user_input], columns=X.columns)
user_scaled = scaler.transform(user_df)

# Prediction & probability
pred = model.predict(user_scaled)[0]
conf = model.predict_proba(user_scaled)[0][pred]

# Output label
label = "Yes \u2013 Likely Winner" if pred == 1 else "No \u2013 Unlikely to Win"

# Winning chance level logic
if pred == 1:
    if conf > 0.8:
        chance = "\U0001f7e2 High Winning Chance"
    elif conf > 0.5:
        chance = "\U0001f7e1 Medium Winning Chance"
    else:
        chance = "\U0001f7e0 Low Winning Chance"
else:
    if conf > 0.8:
        chance = "\U0001f534 Very Low Winning Chance"
    elif conf > 0.5:
        chance = "\U0001f7e0 Low Winning Chance"
    else:
        chance = "\U0001f7e1 Borderline \u2013 Some Chance"

# Display results
print("\n\U0001f50d Runner Winning Prediction Result:")
print(f"\U0001f3c5 Prediction    : {label}")
print(f"\U0001f3af Confidence    : {conf * 100:.2f}%")
print(f"\U0001f4ca Winning Chance: {chance}")

# SHAP Explanation
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(user_scaled)

shap.initjs()
shap.force_plot(explainer.expected_value, shap_values[0], user_df.iloc[0])

In [ ]:
!pip install fpdf --quiet

**Generate and Download Runner Prediction Report**

In [ ]:
!pip install fpdf shap matplotlib --quiet

from fpdf import FPDF
from datetime import datetime
import unicodedata
from matplotlib import pyplot as plt
import shap
import numpy as np

# CLEAN TEXT FUNCTION
def clean_text(text):
    if isinstance(text, str):
        text = unicodedata.normalize("NFKD", text)
        text = text.encode("ascii", "ignore").decode("ascii")
        return text.replace("\u2013", "-").replace("\u2014", "-")
    return str(text)

# SHAP EXPLANATION FUNCTION
def get_shap_explanation(shap_values, feature_names, data_values, prediction, confidence):
    abs_shap = np.abs(shap_values)
    top_indices = abs_shap.argsort()[-3:][::-1]

    explanation = []

    explanation.append("AI Decision Explanation:")
    explanation.append("")
    explanation.append("This chart shows which runner factors influenced the AI's prediction:")
    explanation.append("  Red bars = factors that INCREASE winning likelihood")
    explanation.append("  Blue bars = factors that DECREASE winning likelihood")
    explanation.append("  Longer bars = stronger influence on the decision")
    explanation.append("")

    explanation.append("Key Factors in This Decision:")
    for i, idx in enumerate(top_indices, 1):
        feature = feature_names[idx]
        shap_val = shap_values[idx]
        actual_val = data_values[idx]
        direction = "increases" if shap_val > 0 else "decreases"
        explanation.append(f"{i}. {feature} (value: {actual_val}): {direction} prediction by {abs(shap_val):.3f}")

    explanation.append("")

    conf_pct = confidence * 100
    if conf_pct >= 90:
        conf_level = "Very High - strong evidence for this prediction"
    elif conf_pct >= 70:
        conf_level = "High - good evidence supports this prediction"
    elif conf_pct >= 50:
        conf_level = "Moderate - some uncertainty remains"
    else:
        conf_level = "Low - prediction is uncertain"

    explanation.append(f"Prediction Confidence: {conf_pct:.1f}% ({conf_level})")
    explanation.append("")

    explanation.append("Important Notes:")
    explanation.append("  This AI analysis is a performance screening tool")
    explanation.append("  Should be combined with coach evaluation and race history")
    explanation.append("  Consult sports professionals for final decisions")

    return explanation

# PDF REPORT CLASS
class PDFReport(FPDF):
    def header(self):
        if self.page_no() == 1:
            self.set_font('Arial', 'B', 16)
            self.set_text_color(0, 102, 204)
            self.cell(0, 10, 'Runner Winning Prediction Report', border=False, ln=1, align='C')
            self.ln(5)
            self.set_text_color(0, 0, 0)

    def footer(self):
        self.set_y(-15)
        self.set_font('Arial', 'I', 8)
        self.set_text_color(128)
        self.cell(0, 10, f'Generated on {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}', align='C')

    def add_multiline_text(self, text_lines, font_size=10):
        self.set_font("Arial", size=font_size)
        for line in text_lines:
            if line == "":
                self.ln(3)
            else:
                if any(header in line for header in ["Explanation:", "Decision:", "Factors:", "Notes:", "Confidence:"]):
                    self.set_font("Arial", 'B', font_size)
                    self.cell(0, 6, clean_text(line), ln=1)
                    self.set_font("Arial", size=font_size)
                else:
                    self.cell(0, 6, clean_text(line), ln=1)

# PDF GENERATION FUNCTION
def generate_pdf_report(data, prediction, confidence, chance, name, model, shap_values_user, expected_value):
    pdf = PDFReport()
    pdf.add_page()
    pdf.set_font("Arial", size=12)

    pdf.set_font("Arial", 'B', 14)
    pdf.cell(0, 10, f'Runner Name: {clean_text(name)}', ln=1)
    pdf.ln(3)

    pdf.set_font("Arial", 'B', 12)
    pdf.cell(0, 10, 'Runner Data:', ln=1)
    pdf.set_font("Arial", size=12)

    for col, val in data.items():
        pdf.cell(60, 10, f"{clean_text(col)}:", ln=0)
        pdf.cell(0, 10, f"{clean_text(str(val))}", ln=1)

    pdf.ln(5)

    pdf.set_font("Arial", 'B', 14)
    pdf.cell(0, 10, 'Prediction Summary:', ln=1)
    pdf.set_font("Arial", size=12)

    pdf.cell(60, 10, "Prediction:", ln=0)
    pdf.cell(0, 10, clean_text(prediction), ln=1)

    pdf.cell(60, 10, "Confidence:", ln=0)
    pdf.cell(0, 10, f"{confidence * 100:.2f}%", ln=1)

    pdf.cell(60, 10, "Winning Chance:", ln=0)
    pdf.cell(0, 10, clean_text(chance.replace("\U0001f7e2","High").replace("\U0001f7e1","Medium").replace("\U0001f7e0","Low").replace("\U0001f534","Very Low")), ln=1)

    user_explanation = shap.Explanation(
        values=shap_values_user[0],
        base_values=expected_value,
        data=list(data.values()),
        feature_names=list(data.keys())
    )

    plt.figure(figsize=(8, 4))
    shap.plots.bar(user_explanation, show=False)
    plt.title(f"SHAP Explainability - {clean_text(name)}")
    shap_file = f"shap_plot_{datetime.now().strftime('%H%M%S')}.png"
    plt.savefig(shap_file, bbox_inches='tight')
    plt.close()

    pdf.add_page()
    pdf.ln(10)

    shap_explanation = get_shap_explanation(
        shap_values_user[0],
        list(data.keys()),
        list(data.values()),
        prediction,
        confidence
    )

    pdf.add_multiline_text(shap_explanation, font_size=10)
    pdf.ln(10)

    pdf.set_font("Arial", 'B', 12)
    pdf.cell(0, 10, 'Feature Importance Chart:', ln=1)
    pdf.image(shap_file, x=None, w=180)

    pdf.ln(5)
    pdf.set_font("Arial", 'B', 10)
    pdf.cell(0, 6, "How to Read This Chart:", ln=1)
    pdf.set_font("Arial", size=9)
    interpretation = [
        "  Each bar shows one runner factor (like sprint time, VO2 Max, etc.)",
        "  Bar length = how much that factor influenced the AI's decision",
        "  Red bars push toward Winning prediction",
        "  Blue bars push toward Not Winning prediction",
        "  The AI combines all these factors to make its final prediction"
    ]
    for line in interpretation:
        pdf.cell(0, 5, clean_text(line), ln=1)

    file_name = f"runner_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf"
    pdf.output(file_name)
    print(f"\n\U0001f4c4 PDF report generated: {file_name}")
    return file_name

In [ ]:
from google.colab import files

runner_name = input("Enter runner's name for the report: ")

explainer = shap.TreeExplainer(model)
shap_values_user = explainer.shap_values(user_scaled)
expected_value = explainer.expected_value

pdf_file = generate_pdf_report(
    data=user_df.iloc[0].to_dict(),
    prediction=label,
    confidence=conf,
    chance=chance,
    name=runner_name,
    model=model,
    shap_values_user=shap_values_user,
    expected_value=expected_value
)

# Download the PDF file
files.download(pdf_file)